In [1]:
import pymongo as pm
import pandas as pd

client = pm.MongoClient(
 'bigdatadb.polito.it',
 ssl=True,
 authSource='carsharing',
 username='ictts',
 password='Ict4SM22!',
 tlsAllowInvalidCertificates=True
)

In [2]:
db= client['carsharing']

In [8]:
pb_coll = db['ictts_PermanentBookings']
pb_coll_enjoy= db['ictts_enjoy_PermanentBookings']

In [9]:
pipe = [
    {
        "$project": {
            "hour": {"$hour": "$init_date"},  
            "day": {"$dayOfWeek": "$init_date"},
            "init_location": 1,
            "final_location": 1,
        }
    },
    {
        "$match": {
            "day": {"$gte":1,"$lte":6},
            "hour": {"$gte":0,"$lte":12}
        }
    },
    {
        "$count": "tot"
    }
]

pb_coll.aggregate(pipe)

In [10]:
print(pb_coll.find_one())

{'_id': ObjectId('5a3bb3af20c3dd47cf59003d'), 'init_fuel': 53, 'vendor': 'car2go', 'exterior': 'GOOD', 'vin': 'WME4533421K152664', 'final_address': 'Corso Regina Margherita, 107, 10122 Torino TO', 'engineType': 'CE', 'final_date': datetime.datetime(2017, 11, 1, 0, 19, 45), 'public_transport': {'duration': -1, 'arrival_time': -1, 'distance': -1, 'arrival_date': -1}, 'final_loc': {'coordinates': [7.68634, 45.07567], 'type': 'Point'}, 'smartPhoneRequired': True, 'init_time': 1509490770, 'interior': 'GOOD', 'init_address': 'Lungo Dora Napoli, 18A, 10152 Torino TO', 'init_loc': {'coordinates': [7.68486, 45.08276], 'type': 'Point'}, 'driving': {'duration': 234, 'distance': 966}, 'final_time': 1509491985, 'plate': '288/FF214SJ', 'city': 'Torino', 'final_fuel': 53, 'init_date': datetime.datetime(2017, 10, 31, 23, 59, 30), 'walking': {'duration': 746, 'distance': 982}}


# Q1


In [7]:
Q= 0.90

def load_wtime(db, coll, carsharing=True):
    if carsharing:
        pipe = [
            {
                "$match": {
                    "driving.duration": {"$gt": 0},
                    "driving.distance": {"$gt": 0}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "init_time": 1,
                    "final_time": 1,
                    "duration": "$driving.duration",
                    "day": {
                        "$dateToString": {
                            "date": "$init_date",
                            "format": "%Y-%m-%d"
                        }
                    },
                    "week": {"$isoWeek": "$init_date"},
                    "iso_year": {"$isoWeekYear": "$init_date"},
                    "dow": {"$isoDayOfWeek": "$init_date"},
                    "hour": {"$hour": "$init_date"},
                    "vendor": 1,
                    "init_address": 1,
                    "final_address": 1,
                    "plate": 1,
                    "init_loc": 1,
                    "final_loc": 1,
                    "driving": 1,
                    "walking": 1
                }
            }
        ]

    else:
        raise NotImplementedError(
            "Non-carsharing data processing is not implemented yet."
        )

    collection = db[coll] if isinstance(coll, str) else coll
    df = pd.DataFrame(collection.aggregate(pipe, allowDiskUse=True))

    if df.empty:
        print("Warning: MongoDB query returned no documents.")
        return df

    df["duration_min"] = df["duration"] / 60

    df["driving_distance_km"] = df["driving"].apply(
        lambda x: x["distance"] / 1000
        if isinstance(x, dict) else np.nan
    )

    df["walking_duration_min"] = df["walking"].apply(
        lambda x: x["duration"] / 60
        if isinstance(x, dict) else np.nan
    )

    return df



def best_zone(x, q=Q, log=True, threshold=False, min_duration=1.0):
    x = np.sort(np.asarray(x, dtype=float))
    if threshold:
        x = x[x >= min_duration]
    else:
        x = x[x >= 0]
    n = len(x)
    if n == 0:
        return np.nan, np.nan
    k = int(np.ceil(q * n))
    t = np.log(x) if log else x
    widths = t[k - 1:] - t[:n - k + 1]
    i = np.argmin(widths)
    return x[i], x[i + k - 1]


def keep_best_zone(df, q=Q, col="duration_min", log=True, threshold=False):
    a, b = best_zone(df[col].values, q, log, threshold)
    return df[df[col].between(a, b)].copy(), (a, b)


def get_filtered_data(db,coll,q=Q,carsharing=True,verbose=True,return_extra=False):
    raw = load_wtime(db=db,coll=coll,carsharing=carsharing)

    if raw.empty:
        raise ValueError("No data returned by MongoDB.")

    filtered, zone = keep_best_zone(raw, q=q)

    if verbose:
        a, b = zone
        n0, n1 = len(raw), len(filtered)
        percentage = n1 / n0 if n0 else 0

        print(
            f"zone = [{a:.1f}, {b:.1f}] min "
            f"-> {n1}/{n0} gardés ({percentage:.1%})"
        )

    if return_extra:
        return filtered, {"raw": raw, "zone": zone}

    return filtered

In [11]:
bookings = load_wtime("ictts_PermanentBookings")
bookings_Enjoy = load_wtime("ictts_PermanentBookings")

TypeError: load_wtime() missing 1 required positional argument: 'coll'